# 14b - Compute Human Evaluation Skenario

Notebook ini hanya menghitung evaluasi manusia dari tiga evaluator. Seluruh generasi, penilaian RAGAS, ringkasan RAGAS, dan unggahan keluarannya sudah selesai di notebook 14a.

Tiga dimensi penilaian manusia sama dengan notebook 13: `fluency`, `factual_correctness`, dan `completeness` pada skala ordinal 1–5.


In [ ]:
%pip install -q krippendorff huggingface_hub

import json
import os
from pathlib import Path

import krippendorff
import numpy as np
import pandas as pd
from huggingface_hub import HfApi


## 1. Konfigurasi

In [ ]:
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_Faiss")
    except Exception:
        token = os.environ.get("HF_Faiss")
        if not token:
            raise RuntimeError("Secret HF_Faiss tidak ditemukan.")
        return token


HF_TOKEN = get_hf_token()
HF_REPO = "Makaareeem/publikasi-rag-finetuning-dataset"
OUTPUT_DIR = Path(os.environ.get(
    "OUTPUT_DIR",
    "/kaggle/working" if Path("/kaggle/working").is_dir() else "./scenario_human_summary",
))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RATER_NAMES = ["evaluator_1", "evaluator_2", "evaluator_3"]
DIMENSIONS = ["fluency", "factual_correctness", "completeness"]
SCORE_DOMAIN = [1, 2, 3, 4, 5]


## 2. Ambil kunci dan tiga lembar evaluator dari Hugging Face

In [ ]:
def hf_csv(filename, **kwargs):
    return pd.read_csv(
        f"hf://datasets/{HF_REPO}/{filename}",
        storage_options={"token": HF_TOKEN},
        **kwargs,
    )


key = hf_csv("scenario_human_eval_key.csv", dtype=str, keep_default_na=False)
evaluator_frames = []
for index, evaluator_name in enumerate(RATER_NAMES, start=1):
    filename = f"scenario_human_eval_evaluator{index}_filled.csv"
    frame = hf_csv(filename, dtype=str, keep_default_na=False)
    frame["evaluator"] = evaluator_name
    evaluator_frames.append(frame)
    print(f"Loaded {filename}: {len(frame)} item")


## 3. Validasi penilaian

In [ ]:
required_key = {
    "eval_id", "scenario_id", "scenario", "variant", "model_key",
    "base_question_id", "style_group_id", "style", "typo_group_id", "typo_severity",
}
missing = required_key - set(key.columns)
if missing:
    raise ValueError(f"Kunci kehilangan kolom: {sorted(missing)}")
if not key["eval_id"].is_unique:
    raise ValueError("eval_id pada key tidak unik.")

validated = []
expected_ids = set(key["eval_id"])
for evaluator_name, frame in zip(RATER_NAMES, evaluator_frames):
    missing_columns = {"eval_id", *DIMENSIONS} - set(frame.columns)
    if missing_columns:
        raise ValueError(f"{evaluator_name} kehilangan kolom: {sorted(missing_columns)}")
    if frame["eval_id"].duplicated().any():
        raise ValueError(f"eval_id {evaluator_name} tidak unik.")
    if set(frame["eval_id"]) != expected_ids:
        raise ValueError(f"Daftar eval_id {evaluator_name} tidak sama dengan key.")
    for dimension in DIMENSIONS:
        frame[dimension] = pd.to_numeric(frame[dimension], errors="coerce")
        if frame[dimension].isna().any() or not frame[dimension].isin(SCORE_DOMAIN).all():
            raise ValueError(f"Nilai {dimension} milik {evaluator_name} harus bilangan bulat 1–5.")
    validated.append(frame[["eval_id", "evaluator", *DIMENSIONS]])

human_scores = pd.concat(validated, ignore_index=True).merge(
    key,
    on="eval_id",
    validate="many_to_one",
)
print("Penilaian tervalidasi:", len(human_scores))


## 4. Krippendorff's alpha ordinal per dimensi

In [ ]:
alpha_rows = []
for dimension in DIMENSIONS:
    matrix = human_scores.pivot(
        index="evaluator",
        columns="eval_id",
        values=dimension,
    ).to_numpy(dtype=float)
    try:
        alpha = float(krippendorff.alpha(
            reliability_data=matrix,
            level_of_measurement="ordinal",
        ))
    except ValueError:
        alpha = np.nan
    interpretation = (
        "reliabel kuat" if alpha >= 0.80
        else "dapat diterima" if alpha >= 0.67
        else "tidak memadai"
    ) if pd.notna(alpha) else "tidak dapat dihitung"
    alpha_rows.append({
        "dimension": dimension,
        "alpha_ordinal": alpha,
        "interpretation": interpretation,
        "n_evaluators": len(RATER_NAMES),
    })

reliability = pd.DataFrame(alpha_rows)
display(reliability)


## 5. Ringkasan skor manusia

In [ ]:
def summarize_human(frame, groups):
    summary = frame.groupby(groups, dropna=False)[DIMENSIONS].agg(["mean", "count", "std", "median"])
    summary.columns = ["_".join(part for part in column if part) for column in summary.columns]
    return summary.reset_index()


human_items = human_scores.groupby(
    [
        "eval_id", "scenario_id", "model_key", "scenario", "variant",
        "base_question_id", "style_group_id", "style", "typo_group_id", "typo_severity",
    ],
    as_index=False,
)[DIMENSIONS].mean()

human_summary_by_model = summarize_human(human_items, ["model_key"])
human_summary_by_scenario = summarize_human(human_items, ["model_key", "scenario"])
display(human_summary_by_model)
display(human_summary_by_scenario)


## 6. Perbandingan gaya dan tingkat kesalahan ketik

In [ ]:
style_human = human_items[human_items["scenario"].eq("3_variasi_gaya_bahasa")].copy()
formal = style_human[style_human["style"].eq("formal")][
    ["style_group_id", "model_key", *DIMENSIONS]
].rename(columns={dimension: f"formal_{dimension}" for dimension in DIMENSIONS})
style_human = style_human.merge(formal, on=["style_group_id", "model_key"], validate="many_to_one")
for dimension in DIMENSIONS:
    style_human[f"delta_{dimension}_vs_formal"] = (
        style_human[dimension] - style_human[f"formal_{dimension}"]
    )

clean = human_items[human_items["scenario"].eq("1_faktual_berbasis_narasi")][
    ["base_question_id", "model_key", *DIMENSIONS]
].rename(columns={dimension: f"clean_{dimension}" for dimension in DIMENSIONS})
typo_human = human_items[human_items["scenario"].eq("5_kesalahan_ketik")].merge(
    clean,
    on=["base_question_id", "model_key"],
    validate="one_to_one",
)
for dimension in DIMENSIONS:
    typo_human[f"delta_{dimension}_vs_clean"] = (
        typo_human[dimension] - typo_human[f"clean_{dimension}"]
    )

style_human_summary = style_human.groupby(["model_key", "style"], as_index=False)[
    [f"delta_{dimension}_vs_formal" for dimension in DIMENSIONS]
].mean()
typo_human_summary = typo_human.groupby(["model_key", "typo_severity"], as_index=False)[
    [f"delta_{dimension}_vs_clean" for dimension in DIMENSIONS]
].mean()
display(style_human_summary)
display(typo_human_summary)


## 7. Simpan dan unggah hasil human evaluation

In [ ]:
human_outputs = {
    "scenario_human_scores_combined.csv": human_scores,
    "scenario_human_item_scores.csv": human_items,
    "scenario_krippendorff_alpha.csv": reliability,
    "scenario_human_summary_by_model.csv": human_summary_by_model,
    "scenario_human_summary_by_scenario.csv": human_summary_by_scenario,
    "scenario_human_style_comparison.csv": style_human,
    "scenario_human_style_summary.csv": style_human_summary,
    "scenario_human_typo_comparison.csv": typo_human,
    "scenario_human_typo_summary.csv": typo_human_summary,
}

api = HfApi(token=HF_TOKEN)
for filename, frame in human_outputs.items():
    output_path = OUTPUT_DIR / filename
    frame.to_csv(output_path, index=False)
    api.upload_file(
        path_or_fileobj=str(output_path),
        path_in_repo=filename,
        repo_id=HF_REPO,
        repo_type="dataset",
    )
    print(f"Terunggah: {HF_REPO}/{filename}")

summary_json = {
    "evaluation_type": "human_only",
    "dimensions": DIMENSIONS,
    "evaluators": RATER_NAMES,
    "n_items": int(len(key)),
    "krippendorff_alpha": {
        row.dimension: None if pd.isna(row.alpha_ordinal) else float(row.alpha_ordinal)
        for row in reliability.itertuples()
    },
}
json_path = OUTPUT_DIR / "scenario_human_evaluation_summary.json"
json_path.write_text(json.dumps(summary_json, ensure_ascii=False, indent=2), encoding="utf-8")
api.upload_file(
    path_or_fileobj=str(json_path),
    path_in_repo=json_path.name,
    repo_id=HF_REPO,
    repo_type="dataset",
)
print("Selesai: notebook 14b hanya menghasilkan human evaluation.")
